In [36]:
import pandas as pd
import re

In [37]:
REGION = 'ExtNord'
enc = 'utf-8'

In [38]:
data = pd.read_csv(f'../../data/{REGION}/CM_{REGION}_Landcover_2001-2023_original.csv', encoding=enc)
data.columns = data.columns.str.lower()

In [39]:
data.head()

,x,y,2001_01_01_lc_prop1,2001_01_01_lc_prop1_assessment,2001_01_01_lc_prop2,2001_01_01_lc_prop2_assessment,2001_01_01_lc_prop3,2001_01_01_lc_prop3_assessment,2001_01_01_lc_type1,2001_01_01_lc_type2,...,2023_01_01_lc_prop2_assessment,2023_01_01_lc_prop3,2023_01_01_lc_prop3_assessment,2023_01_01_lc_type1,2023_01_01_lc_type2,2023_01_01_lc_type3,2023_01_01_lc_type4,2023_01_01_lc_type5,2023_01_01_lw,2023_01_01_qc
0,14.40633,10.67157,31,95,36,94.0,30,95,12,12,...,99.0,30,99,12,12,1,6,7,2,0
1,14.44281,10.67157,31,97,36,96.0,30,97,12,12,...,99.0,30,99,12,12,1,6,7,2,0
2,14.44281,10.70751,31,96,36,95.0,30,96,12,12,...,99.0,30,99,12,12,1,6,7,2,0
3,14.44281,10.74344,31,98,36,98.0,30,98,12,12,...,99.0,30,99,12,12,1,6,7,2,0
4,14.47929,10.70751,31,96,36,96.0,30,96,12,12,...,99.0,30,99,12,12,1,6,7,2,0


In [40]:
lc_columns = data.columns.tolist()
lc_columns = [x.split('_')[0] for x in lc_columns]

lc_columns
lc_years = set()

for i in range(0, len(lc_columns)):
    try:
        lc_years.add(int(lc_columns[i]))
    except ValueError:
        pass

lc_years = list(lc_years)

In [41]:
dates_ = re.compile(r"[0-9]{4}_[0-9]{2}_[0-9]{2}_")
landcover_datasets_per_year = []

for year in lc_years:
    frame_name = f'landcover_{year}'
    location_columns = ['x', 'y']
    filtered_columns = [col for col in data if col.startswith(f'{year}')]
    columns_to_keep = location_columns + filtered_columns
    locals()[frame_name] = data[columns_to_keep].copy()
    locals()[frame_name].insert(2, 'year', int(year))
    locals()[frame_name] = locals()[frame_name].rename(columns=lambda x: re.sub(dates_,'',x))
    landcover_datasets_per_year.append(locals()[frame_name])

In [42]:
landcover_2024 = landcover_2023.copy()
landcover_2024['year'] = 2024
landcover_datasets_per_year.append(landcover_2024)

In [43]:
landcover_processed = pd.concat(landcover_datasets_per_year, ignore_index=True)
landcover_processed.reset_index(drop=True, inplace=True)

In [44]:
landcover_processed

,x,y,year,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc
0,14.40633,10.67157,2001,31,95,36,94.0,30,95,12,12,1,6,7,2,0
1,14.44281,10.67157,2001,31,97,36,96.0,30,97,12,12,1,6,7,2,0
2,14.44281,10.70751,2001,31,96,36,95.0,30,96,12,12,1,6,7,2,0
3,14.44281,10.74344,2001,31,98,36,98.0,30,98,12,12,1,6,7,2,0
4,14.47929,10.70751,2001,31,96,36,96.0,30,96,12,12,1,6,7,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49699,14.00505,10.74344,2024,31,99,36,99.0,30,99,12,12,1,6,7,2,0
49700,14.00505,10.77937,2024,31,99,36,99.0,30,99,12,12,1,6,7,2,0
49701,14.00505,10.85124,2024,31,99,36,99.0,30,99,12,12,1,6,7,2,0
49702,14.04153,10.74344,2024,31,99,36,99.0,30,99,12,12,1,6,7,2,0


In [45]:
landcover_processed.to_csv(f'../../data/{REGION}/CM_{REGION}_Landcover_2001-2024_processed.csv', index=False, encoding=enc)